In [1]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch dataset (id=938)
regensburg_pediatric_appendicitis = fetch_ucirepo(id=938)

# Features and targets as pandas DataFrames
X = regensburg_pediatric_appendicitis.data.features
y = regensburg_pediatric_appendicitis.data.targets

# Combine into one DataFrame
df = pd.concat([X, y], axis=1)

print(df.shape)
print(df.head())

(782, 56)
     Age   BMI     Sex  Height  Weight  Length_of_Stay  Alvarado_Score  \
0  12.68  16.9  female   148.0    37.0             3.0             4.0   
1  14.10  31.9    male   147.0    69.5             2.0             5.0   
2  14.14  23.3  female   163.0    62.0             4.0             5.0   
3  16.37  20.6  female   165.0    56.0             3.0             7.0   
4  11.08  16.9  female   163.0    45.0             3.0             5.0   

   Paedriatic_Appendicitis_Score Appendix_on_US  Appendix_Diameter  ...  \
0                            3.0            yes                7.1  ...   
1                            4.0             no                NaN  ...   
2                            3.0             no                NaN  ...   
3                            6.0             no                NaN  ...   
4                            6.0            yes                7.0  ...   

  Bowel_Wall_Thickening Conglomerate_of_Bowel_Loops Ileus Coprostasis  \
0                   N

In [2]:
# check data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 56 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Age                               781 non-null    float64
 1   BMI                               755 non-null    float64
 2   Sex                               780 non-null    str    
 3   Height                            756 non-null    float64
 4   Weight                            779 non-null    float64
 5   Length_of_Stay                    778 non-null    float64
 6   Alvarado_Score                    730 non-null    float64
 7   Paedriatic_Appendicitis_Score     730 non-null    float64
 8   Appendix_on_US                    777 non-null    str    
 9   Appendix_Diameter                 498 non-null    float64
 10  Migratory_Pain                    773 non-null    str    
 11  Lower_Right_Abd_Pain              774 non-null    str    
 12  Contralateral_Rebou

In [13]:
# percentage of missing values per column
missing_percent = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_percent)


Abscess_Location                    98.333333
Gynecological_Findings              96.666667
Conglomerate_of_Bowel_Loops         94.487179
Segmented_Neutrophils               93.076923
Ileus                               92.307692
Perfusion                           91.923077
Enteritis                           91.538462
Appendicolith                       91.153846
Coprostasis                         90.897436
Perforation                         89.615385
Appendicular_Abscess                89.102564
Bowel_Wall_Thickening               87.307692
Lymph_Nodes_Location                84.487179
Target_Sign                         82.307692
Meteorism                           82.051282
Pathological_Lymph_Nodes            73.974359
Appendix_Wall_Layers                72.051282
Surrounding_Tissue_Reaction         67.692308
Appendix_Diameter                   36.153846
RBC_in_Urine                        26.153846
Ketones_in_Urine                    25.384615
WBC_in_Urine                      

In [14]:
# Drop rows with missing target (Diagnosis)
df = df.dropna(subset = ["Diagnosis"])

print(df.shape)
print(df["Diagnosis"].value_counts())

(780, 56)
Diagnosis
appendicitis       463
no appendicitis    317
Name: count, dtype: int64


In [11]:
# Count fully duplicated rows
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [15]:
# Drop columns with more than 70% missing values

high_missing = missing_percent[missing_percent > 70].index.tolist()
print(len(high_missing), high_missing)

# Drop them
df = df.drop(columns=high_missing)

print(df.shape)

17 ['Abscess_Location', 'Gynecological_Findings', 'Conglomerate_of_Bowel_Loops', 'Segmented_Neutrophils', 'Ileus', 'Perfusion', 'Enteritis', 'Appendicolith', 'Coprostasis', 'Perforation', 'Appendicular_Abscess', 'Bowel_Wall_Thickening', 'Lymph_Nodes_Location', 'Target_Sign', 'Meteorism', 'Pathological_Lymph_Nodes', 'Appendix_Wall_Layers']
(780, 39)


In [16]:
# Management and Severity are outcomes decided after diagnosis, so they leak the answer
X = df.drop(columns=["Diagnosis", "Management", "Severity"])
y = df["Diagnosis"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (780, 36)
Target shape: (780,)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Training rows: 624
Testing rows: 156
Diagnosis
appendicitis       0.592949
no appendicitis    0.407051
Name: proportion, dtype: float64
Diagnosis
appendicitis       0.596154
no appendicitis    0.403846
Name: proportion, dtype: float64
